# Task #3: Validate risk scoring against advisor rankings
Self-contained: rebuilds the team's test split, scores clauses with a transparent rule-based
scorer built from the advisor's stated risk signals, and compares against the advisor's 1–5 rankings.

**Before running:** upload `advisor_ranking_sheet.csv` to the Colab Files panel (left sidebar).

In [ ]:
import json, re, os
import numpy as np, pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import cohen_kappa_score, confusion_matrix

REPO = "/content/team_repo"
if not os.path.exists(REPO):
    os.system(f"git clone -q --depth 1 https://github.com/Break-Through-Tech/Accenture-1K-contract-review-challenge.git {REPO}")
DATA = f"{REPO}/data"
LEVELS = ["Low", "Medium", "High"]

## 1. Rebuild the team's test split (same contract-level 80/10/10, seed 42)

In [ ]:
train = json.load(open(f"{DATA}/train_separate_questions.json"))["data"]
test = json.load(open(f"{DATA}/test.json"))["data"]
contracts = {c["title"]: c for c in train + test}
ids = np.array(list(dict.fromkeys(c["title"] for c in train + test)), dtype=object)
rng = np.random.default_rng(42); rng.shuffle(ids)
n = len(ids); test_ids = list(ids[int(n*.8) + int(n*.1):])
cat = lambda qa: re.sub(r"_\d+$", "", qa["id"].split("__")[-1])

def present_categories(cid):
    """Gold clause categories present in a contract -> {category: longest evidence span}."""
    out = {}
    for qa in contracts[cid]["paragraphs"][0]["qas"]:
        if qa["answers"]:
            out[cat(qa)] = max((a["text"] for a in qa["answers"]), key=len).strip()
    return out
print("Test contracts:", len(test_ids))

## 2. Rule-based risk scorer
Signals come straight from the advisor's Q&A in `Challenge-Project-Overview.md`:
uncapped liability = automatic High; liability cap with no amount; one-way termination;
auto-renewal with no notice window; IP assignment; missing priority clause = at least Medium.
Every weight and threshold is a parameter so the sensitivity analysis can vary them.

In [ ]:
DEFAULT_WEIGHTS = {
    "uncapped_liability": 90, "cap_without_amount": 55, "cap_with_amount": 15,
    "one_way_termination": 70, "mutual_termination": 25,
    "renewal_no_notice_window": 70, "renewal_with_notice": 25,
    "ip_assignment": 50, "change_of_control": 50, "non_compete": 55,
    "most_favored_nation": 50, "audit_rights": 20, "governing_law": 10,
    "missing_priority_clause": 55,
}
DEFAULT_THRESHOLDS = {"medium": 35, "high": 65}
AUTO_HIGH = {"uncapped_liability"}            # advisor: High regardless of anything else
MONEY = re.compile(r"\$|dollars|usd|fees? (paid|payable)|amounts? paid|\d[\d,]*\s*(million|thousand)", re.I)
MUTUAL = re.compile(r"\b(either party|both parties|each party|any party)\b", re.I)

def clause_signals(category, text, contract_cats):
    if text is None:                                     # clause absent from contract
        return ["missing_priority_clause"]
    if category == "Uncapped Liability":            return ["uncapped_liability"]
    if category == "Cap On Liability":
        return ["cap_with_amount" if MONEY.search(text) else "cap_without_amount"]
    if category == "Termination For Convenience":
        return ["mutual_termination" if MUTUAL.search(text) else "one_way_termination"]
    if category in ("Renewal Term", "Notice Period To Terminate Renewal"):
        if not re.search(r"automatic|auto-renew|shall renew|will renew|successive", text, re.I):
            return ["renewal_with_notice"]          # optional/negotiated renewal → low risk
        return ["renewal_with_notice" if "Notice Period To Terminate Renewal" in contract_cats
                else "renewal_no_notice_window"]
    return [{"Ip Ownership Assignment": "ip_assignment", "Change Of Control": "change_of_control",
             "Non-Compete": "non_compete", "Most Favored Nation": "most_favored_nation",
             "Audit Rights": "audit_rights", "Governing Law": "governing_law"}.get(category, "governing_law")]

def score_clause(category, text, contract_cats, weights=DEFAULT_WEIGHTS, thresholds=DEFAULT_THRESHOLDS):
    sig = clause_signals(category, text, contract_cats)
    score = max(weights[s] for s in sig)
    if any(s in AUTO_HIGH for s in sig):
        level = "High"
    else:
        level = "High" if score >= thresholds["high"] else "Medium" if score >= thresholds["medium"] else "Low"
    return score, level, ", ".join(sig)

### Team scorer (Naavya & Yutika's `RISK_CONFIG`, applied one clause at a time)
Copied from the risk-scorer notebook. Keep in sync if they change it.

In [ ]:
RISK_CONFIG = {
    "weights": {"Uncapped Liability": 35, "Non-Compete": 25, "Exclusivity": 15, "Anti-Assignment": 10,
                "Termination For Convenience": 15, "Ip Ownership Assignment": 20,
                "Minimum Commitment": 15, "Liquidated Damages": 20},
    "overrides": {"Uncapped Liability": 85, "Non-Compete": 75},
    "required_clauses": {"Governing Law": 10, "Cap On Liability": 25,
                         "Termination For Convenience": 10, "Audit Rights": 5},
    "levels": {"medium": 35, "high": 70},
}

def score_clause_team(category, text, contract_cats, config=RISK_CONFIG):
    if text is None:                                   # missing clause
        s = config["required_clauses"].get(category, 0)
    else:
        s = max(config["weights"].get(category, 0), config["overrides"].get(category, 0))
    lv = config["levels"]
    level = "High" if s >= lv["high"] else "Medium" if s >= lv["medium"] else "Low"
    return s, level, "team_rules"

## 3. Load the advisor sheet and score every item

In [ ]:
sheet = pd.read_csv("advisor_ranking_sheet.csv")   # columns: item_id, contract_id, category, excerpt, advisor_score_1to5, advisor_notes

def score_sheet(sheet, scorer=score_clause, **kwargs):
    """Score every sheet item with `scorer` (score_clause or score_clause_team)."""
    out = []
    for r in sheet.itertuples():
        cats = present_categories(r.contract_id)
        text = None if str(r.excerpt).startswith("[Contract has NO") else r.excerpt
        out.append(scorer(r.category, text, cats, **kwargs))
    return pd.DataFrame(out, columns=["risk_score", "risk_level", "signals_fired"], index=sheet.index)

df = pd.concat([sheet, score_sheet(sheet)], axis=1)                     # text-aware rules
team = score_sheet(sheet, scorer=score_clause_team)                      # team RISK_CONFIG

# ---- PLACEHOLDER: delete once the advisor's scores are in the sheet ----
if df["advisor_score_1to5"].isna().all():
    print("⚠️ No advisor scores yet: using random placeholder scores so the code runs.")
    df["advisor_score_1to5"] = np.random.default_rng(0).integers(1, 6, len(df))
# -----------------------------------------------------------------------
df["advisor_level"] = pd.cut(df.advisor_score_1to5, [0, 2, 3, 5], labels=LEVELS).astype(str)
df_team = df.assign(risk_score=team.risk_score, risk_level=team.risk_level, signals_fired=team.signals_fired)
df[["item_id", "category", "signals_fired", "risk_score", "risk_level", "advisor_score_1to5", "advisor_level"]]

## 4. Agreement metrics

In [ ]:
def metrics(d):
    rho = spearmanr(d.advisor_score_1to5, d.risk_score)[0]
    hi = d.advisor_level == "High"
    return {"spearman_rho": rho,
            "bucket_agreement": (d.advisor_level == d.risk_level).mean(),
            "weighted_kappa": cohen_kappa_score(d.advisor_level, d.risk_level, labels=LEVELS, weights="quadratic"),
            "high_risk_recall": (d.risk_level[hi] == "High").mean() if hi.any() else np.nan}

def full_report(d):
    m = metrics(d)
    boot = [spearmanr(*d.sample(len(d), replace=True, random_state=i)[["advisor_score_1to5", "risk_score"]].T.values)[0]
            for i in range(2000)]
    m["rho_95ci"] = tuple(np.round(np.nanpercentile(boot, [2.5, 97.5]), 3))
    m["level_counts"] = d.risk_level.value_counts().reindex(LEVELS, fill_value=0).to_dict()
    return m

comparison = pd.DataFrame({"text_aware_rules": full_report(df), "team_RISK_CONFIG": full_report(df_team)})
display(comparison)
for name, d in [("text-aware rules", df), ("team RISK_CONFIG", df_team)]:
    print(f"\n{name}: advisor level (rows) vs system level (columns)")
    print(pd.DataFrame(confusion_matrix(d.advisor_level, d.risk_level, labels=LEVELS),
                       index=[f"advisor {l}" for l in LEVELS], columns=[f"system {l}" for l in LEVELS]))

## 5. Sensitivity analysis: is the Medium/High boundary stable?

In [ ]:
base = df.risk_level
rows = []
for hi_t in range(50, 81, 5):                                   # sweep the High threshold
    s = score_sheet(sheet, thresholds={"medium": 35, "high": hi_t})   # text-aware rules
    rows.append({"change": f"high threshold = {hi_t}", "items_changed_level": (s.risk_level != base).sum(),
                 **metrics(df.assign(risk_score=s.risk_score, risk_level=s.risk_level))})
for sig in DEFAULT_WEIGHTS:                                     # each weight ±20%
    for f in (0.8, 1.2):
        w = {**DEFAULT_WEIGHTS, sig: DEFAULT_WEIGHTS[sig] * f}
        s = score_sheet(sheet, weights=w)
        rows.append({"change": f"{sig} x{f}", "items_changed_level": (s.risk_level != base).sum(),
                     **metrics(df.assign(risk_score=s.risk_score, risk_level=s.risk_level))})
sens = pd.DataFrame(rows).round(3)
sens.insert(0, "scorer", "text_aware_rules")

# Same analysis for the team's RISK_CONFIG
import copy
base_t = df_team.risk_level
rows_t = []
for hi_t in range(55, 86, 5):
    cfg = copy.deepcopy(RISK_CONFIG); cfg["levels"]["high"] = hi_t
    s = score_sheet(sheet, scorer=score_clause_team, config=cfg)
    rows_t.append({"change": f"high threshold = {hi_t}", "items_changed_level": (s.risk_level != base_t).sum(),
                   **metrics(df_team.assign(risk_score=s.risk_score, risk_level=s.risk_level))})
for section in ["weights", "overrides", "required_clauses"]:
    for k in RISK_CONFIG[section]:
        for f in (0.8, 1.2):
            cfg = copy.deepcopy(RISK_CONFIG); cfg[section][k] *= f
            s = score_sheet(sheet, scorer=score_clause_team, config=cfg)
            rows_t.append({"change": f"{section}[{k}] x{f}", "items_changed_level": (s.risk_level != base_t).sum(),
                           **metrics(df_team.assign(risk_score=s.risk_score, risk_level=s.risk_level))})
sens_t = pd.DataFrame(rows_t).round(3)
sens_t.insert(0, "scorer", "team_RISK_CONFIG")
sens = pd.concat([sens, sens_t], ignore_index=True)
sens.sort_values("items_changed_level", ascending=False).head(20)

## 6. Error analysis: where the system and advisor disagree

In [ ]:
dis = df[(df.advisor_level != df.risk_level) | ((df.risk_score / 20 - df.advisor_score_1to5).abs() >= 2)]
dis = dis.assign(team_level=df_team.risk_level)[["item_id", "category", "excerpt", "signals_fired", "risk_level",
                                                 "team_level", "advisor_level", "advisor_notes"]].copy()
dis["likely_cause"] = ""   # fill in: missing rule / rule misfired / ambiguous wording / judgment difference
print(len(dis), "disagreements")
dis

In [ ]:
df.assign(team_score=df_team.risk_score, team_level=df_team.risk_level).to_csv("risk_validation_results.csv", index=False)
comparison.to_csv("risk_scorer_comparison.csv")
sens.to_csv("risk_sensitivity_analysis.csv", index=False)
dis.to_csv("risk_disagreements.csv", index=False)